# 20260126 美債殖利率曲線分析
## Bloomberg BQNT 分析報告

### 核心觀點
- **看弱30年期**: 長端受赤字及聯準會獨立性影響
- **看好曲線腹部**: 5s/30s, 10s/30s 利差有望擴大
- **短端偏上**: 市場過度反應鷹派態度，預期將隨通膨緩和而下行

In [ ]:
# Bloomberg BQNT 環境設定
import bql
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# 初始化 BQL 服務
bq = bql.Service()

# 設定中文字體 (Bloomberg 環境)
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'Arial Unicode MS']
plt.rcParams['axes.unicode_minus'] = False

print("Bloomberg BQNT 環境初始化完成")
print(f"分析日期: {datetime.now().strftime('%Y-%m-%d')}")

---
## 1. 美國國債殖利率曲線分析

In [ ]:
# 定義美國國債 Ticker
# 使用 Bloomberg 標準國債代碼
treasury_tickers = {
    '3M': 'GB3 Govt',
    '6M': 'GB6 Govt', 
    '1Y': 'GB12 Govt',
    '2Y': 'USGG2YR Index',
    '3Y': 'USGG3YR Index',
    '5Y': 'USGG5YR Index',
    '7Y': 'USGG7YR Index',
    '10Y': 'USGG10YR Index',
    '20Y': 'USGG20YR Index',
    '30Y': 'USGG30YR Index'
}

# 關鍵曲線價差分析的 Ticker
spread_analysis = {
    '2s10s': ('USGG2YR Index', 'USGG10YR Index'),
    '5s30s': ('USGG5YR Index', 'USGG30YR Index'),
    '10s30s': ('USGG10YR Index', 'USGG30YR Index')
}

print("國債 Ticker 定義完成")

In [ ]:
# 獲取當前殖利率曲線數據
def get_current_yield_curve():
    """
    獲取當前美國國債殖利率曲線
    """
    tickers = list(treasury_tickers.values())
    
    # BQL 查詢當前殖利率
    query = bql.Request(
        tickers,
        {'yield': bq.data.px_last()}
    )
    response = bq.execute(query)
    
    # 整理數據
    yields = {}
    for tenor, ticker in treasury_tickers.items():
        try:
            yields[tenor] = response['yield'].df().loc[ticker, 'value']
        except:
            yields[tenor] = np.nan
    
    return pd.Series(yields, name='Yield (%)')

# 獲取歷史殖利率數據
def get_historical_yields(ticker, days=252):
    """
    獲取歷史殖利率數據
    """
    start_date = (datetime.now() - timedelta(days=days)).strftime('%Y-%m-%d')
    
    query = bql.Request(
        ticker,
        {'yield': bq.data.px_last(dates=bq.func.range(start_date, '0d'))}
    )
    response = bq.execute(query)
    
    return response['yield'].df()

print("數據獲取函數定義完成")

In [ ]:
# 獲取殖利率曲線歷史數據 (過去一年)
key_tenors = ['USGG2YR Index', 'USGG5YR Index', 'USGG10YR Index', 'USGG30YR Index']

# BQL 批量查詢
start_date = (datetime.now() - timedelta(days=365)).strftime('%Y-%m-%d')

query = bql.Request(
    key_tenors,
    {'yield': bq.data.px_last(dates=bq.func.range(start_date, '0d'))}
)

try:
    response = bq.execute(query)
    yield_data = response['yield'].df()
    print("歷史殖利率數據獲取成功")
    print(f"數據期間: {start_date} 至今")
    print(f"數據點數: {len(yield_data)}")
except Exception as e:
    print(f"數據獲取錯誤: {e}")
    # 使用模擬數據進行展示
    print("使用模擬數據進行展示...")

In [ ]:
# 圖表1: 美國國債殖利率曲線 - 當前 vs 歷史比較
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 模擬數據 (實際使用時會從 BQL 獲取)
tenors = ['3M', '6M', '1Y', '2Y', '3Y', '5Y', '7Y', '10Y', '20Y', '30Y']
tenor_numeric = [0.25, 0.5, 1, 2, 3, 5, 7, 10, 20, 30]

# 當前曲線 (2026年1月)
current_curve = [4.35, 4.28, 4.15, 4.05, 4.02, 4.10, 4.25, 4.35, 4.55, 4.65]
# 三個月前曲線
curve_3m_ago = [4.55, 4.48, 4.35, 4.25, 4.18, 4.15, 4.22, 4.30, 4.48, 4.55]
# 六個月前曲線  
curve_6m_ago = [5.25, 5.18, 4.95, 4.55, 4.38, 4.25, 4.28, 4.35, 4.50, 4.58]

# 子圖1: 殖利率曲線比較
ax1 = axes[0, 0]
ax1.plot(tenor_numeric, current_curve, 'b-o', linewidth=2.5, markersize=8, label='當前 (2026/01)')
ax1.plot(tenor_numeric, curve_3m_ago, 'g--s', linewidth=1.5, markersize=6, label='3個月前', alpha=0.7)
ax1.plot(tenor_numeric, curve_6m_ago, 'r-.^', linewidth=1.5, markersize=6, label='6個月前', alpha=0.7)

ax1.set_xlabel('期限 (年)', fontsize=12)
ax1.set_ylabel('殖利率 (%)', fontsize=12)
ax1.set_title('美國國債殖利率曲線比較', fontsize=14, fontweight='bold')
ax1.legend(loc='lower right', fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.set_xlim(0, 32)
ax1.set_ylim(3.8, 5.5)

# 標註關鍵區域
ax1.axvspan(4, 12, alpha=0.15, color='green', label='曲線腹部 (看好)')
ax1.axvspan(25, 32, alpha=0.15, color='red', label='長端 (看弱)')
ax1.annotate('看好區域\n(5Y-10Y)', xy=(7, 4.0), fontsize=10, ha='center', color='green')
ax1.annotate('看弱區域\n(30Y)', xy=(30, 4.8), fontsize=10, ha='center', color='red')

# 子圖2: 曲線變化 (當前 vs 3個月前)
ax2 = axes[0, 1]
curve_change = [c - p for c, p in zip(current_curve, curve_3m_ago)]
colors = ['green' if x < 0 else 'red' for x in curve_change]
bars = ax2.bar(tenors, curve_change, color=colors, alpha=0.7, edgecolor='black')
ax2.axhline(y=0, color='black', linestyle='-', linewidth=0.5)
ax2.set_xlabel('期限', fontsize=12)
ax2.set_ylabel('變化 (bp)', fontsize=12)
ax2.set_title('殖利率變化 (當前 vs 3個月前)', fontsize=14, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')

# 添加數值標籤
for bar, val in zip(bars, curve_change):
    height = bar.get_height()
    ax2.annotate(f'{val*100:.0f}bp',
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3 if height >= 0 else -10),
                textcoords="offset points",
                ha='center', va='bottom' if height >= 0 else 'top',
                fontsize=9)

# 子圖3: 關鍵利差走勢
ax3 = axes[1, 0]
dates = pd.date_range(end=datetime.now(), periods=252, freq='B')

# 模擬利差數據
np.random.seed(42)
spread_5s30s = np.cumsum(np.random.randn(252) * 0.02) + 0.55  # 5s30s 利差
spread_10s30s = np.cumsum(np.random.randn(252) * 0.015) + 0.30  # 10s30s 利差
spread_2s10s = np.cumsum(np.random.randn(252) * 0.025) + 0.30  # 2s10s 利差

ax3.plot(dates, spread_5s30s * 100, 'b-', linewidth=2, label='5s30s', alpha=0.9)
ax3.plot(dates, spread_10s30s * 100, 'g-', linewidth=2, label='10s30s', alpha=0.9)
ax3.plot(dates, spread_2s10s * 100, 'r--', linewidth=1.5, label='2s10s', alpha=0.7)

ax3.set_xlabel('日期', fontsize=12)
ax3.set_ylabel('利差 (bp)', fontsize=12)
ax3.set_title('關鍵曲線利差走勢 (過去一年)', fontsize=14, fontweight='bold')
ax3.legend(loc='upper left', fontsize=10)
ax3.grid(True, alpha=0.3)
ax3.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax3.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
plt.setp(ax3.xaxis.get_majorticklabels(), rotation=45)

# 子圖4: 策略建議摘要
ax4 = axes[1, 1]
ax4.axis('off')

strategy_text = """
╔══════════════════════════════════════════════════════════════╗
║                    殖利率曲線策略建議                          ║
╠══════════════════════════════════════════════════════════════╣
║                                                              ║
║  📊 短端 (2Y以下):                                           ║
║     • 市場過度反應Fed鷹派態度                                  ║
║     • 通膨預期低於預期 → 短端利率有望下行                       ║
║     • 全年預期降息兩碼，6月第一次                              ║
║                                                              ║
║  📈 腹部 (5Y-10Y): [看多]                                    ║
║     • 相對價值較佳                                           ║
║     • 受益於曲線陡峭化交易                                    ║
║                                                              ║
║  📉 長端 (30Y): [看空]                                       ║
║     • 赤字壓力持續                                           ║
║     • 聯準會獨立性風險                                        ║
║     • 買盤結構中期穩定，但供給面仍有壓力                        ║
║                                                              ║
║  🎯 推薦交易:                                                ║
║     • 做多 5s/30s 利差 (目標: +20bp)                         ║
║     • 做多 10s/30s 利差 (目標: +15bp)                        ║
║                                                              ║
╚══════════════════════════════════════════════════════════════╝
"""

ax4.text(0.5, 0.5, strategy_text, transform=ax4.transAxes, fontsize=11,
         verticalalignment='center', horizontalalignment='center',
         fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.savefig('yield_curve_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n圖表1: 殖利率曲線分析已生成")

---
## 2. 通膨數據分析 - 美國 vs 歐洲

In [ ]:
# 通膨相關 Bloomberg Ticker
inflation_tickers = {
    # 美國通膨數據
    'US_CPI_YoY': 'CPI YOY Index',
    'US_Core_CPI_YoY': 'CPUPAXFE Index',
    'US_PCE_YoY': 'PCE DEFY Index',
    'US_Core_PCE_YoY': 'PCE CYOY Index',
    
    # 歐元區通膨數據
    'EU_HICP_YoY': 'ECCPEST Index',
    'EU_Core_HICP_YoY': 'ECCPCORE Index',
    
    # 通膨預期 (Breakeven)
    'US_5Y_Breakeven': 'USGGBE05 Index',
    'US_10Y_Breakeven': 'USGGBE10 Index',
    'EU_5Y_Breakeven': 'EUSWI5 Curncy',
    
    # 能源價格
    'WTI_Crude': 'CL1 Comdty',
    'Brent_Crude': 'CO1 Comdty',
    'Natural_Gas': 'NG1 Comdty'
}

print("通膨相關 Ticker 定義完成")

In [ ]:
# 圖表2: 通膨數據比較 - 美國 vs 歐洲
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 模擬通膨數據
months = pd.date_range(start='2024-01-01', end='2025-12-01', freq='MS')

# 美國 CPI 數據
us_headline_cpi = [3.1, 3.2, 3.5, 3.4, 3.3, 3.0, 2.9, 2.5, 2.4, 2.6, 2.7, 2.9,
                   2.8, 2.6, 2.5, 2.4, 2.3, 2.2, 2.3, 2.4, 2.3, 2.2, 2.1, 2.0]
us_core_cpi = [3.9, 3.8, 3.8, 3.6, 3.4, 3.3, 3.2, 3.2, 3.3, 3.3, 3.3, 3.2,
               3.1, 3.0, 2.9, 2.8, 2.8, 2.7, 2.6, 2.5, 2.4, 2.4, 2.3, 2.3]

# 歐元區 HICP 數據
eu_headline_hicp = [2.8, 2.6, 2.4, 2.4, 2.6, 2.5, 2.6, 2.2, 1.7, 2.0, 2.3, 2.4,
                    2.2, 2.1, 2.0, 1.9, 1.8, 1.9, 2.0, 2.1, 2.0, 1.9, 1.9, 1.9]
eu_core_hicp = [3.3, 3.1, 2.9, 2.7, 2.9, 2.9, 2.9, 2.8, 2.7, 2.7, 2.7, 2.7,
                2.6, 2.5, 2.5, 2.4, 2.4, 2.3, 2.3, 2.3, 2.3, 2.3, 2.3, 2.3]

# 子圖1: 美國通膨走勢
ax1 = axes[0, 0]
ax1.plot(months, us_headline_cpi, 'b-o', linewidth=2, markersize=5, label='總體 CPI')
ax1.plot(months, us_core_cpi, 'r-s', linewidth=2, markersize=5, label='核心 CPI')
ax1.axhline(y=2.0, color='green', linestyle='--', linewidth=2, label='Fed 目標 (2%)')
ax1.fill_between(months, 1.5, 2.5, alpha=0.1, color='green')

ax1.set_xlabel('日期', fontsize=12)
ax1.set_ylabel('年增率 (%)', fontsize=12)
ax1.set_title('美國 CPI 通膨走勢', fontsize=14, fontweight='bold')
ax1.legend(loc='upper right', fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax1.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.setp(ax1.xaxis.get_majorticklabels(), rotation=45)

# 子圖2: 歐元區通膨走勢
ax2 = axes[0, 1]
ax2.plot(months, eu_headline_hicp, 'b-o', linewidth=2, markersize=5, label='總體 HICP')
ax2.plot(months, eu_core_hicp, 'r-s', linewidth=2, markersize=5, label='核心 HICP')
ax2.axhline(y=2.0, color='green', linestyle='--', linewidth=2, label='ECB 目標 (2%)')
ax2.fill_between(months, 1.5, 2.5, alpha=0.1, color='green')

ax2.set_xlabel('日期', fontsize=12)
ax2.set_ylabel('年增率 (%)', fontsize=12)
ax2.set_title('歐元區 HICP 通膨走勢', fontsize=14, fontweight='bold')
ax2.legend(loc='upper right', fontsize=10)
ax2.grid(True, alpha=0.3)
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax2.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45)

# 子圖3: 通膨分項比較 (12月數據)
ax3 = axes[1, 0]

categories = ['總體\n通膨', '核心\n通膨', '商品', '服務', '食品', '能源']
us_dec_data = [2.0, 2.3, 0.2, 3.2, 2.5, -1.5]
eu_dec_data = [1.9, 2.3, 0.4, 3.4, 2.8, -2.0]

x = np.arange(len(categories))
width = 0.35

bars1 = ax3.bar(x - width/2, us_dec_data, width, label='美國', color='steelblue', alpha=0.8)
bars2 = ax3.bar(x + width/2, eu_dec_data, width, label='歐元區', color='darkorange', alpha=0.8)

ax3.axhline(y=2.0, color='green', linestyle='--', linewidth=2, alpha=0.7)
ax3.set_xlabel('分項', fontsize=12)
ax3.set_ylabel('年增率 (%)', fontsize=12)
ax3.set_title('12月通膨分項比較 (美國 vs 歐元區)', fontsize=14, fontweight='bold')
ax3.set_xticks(x)
ax3.set_xticklabels(categories)
ax3.legend(loc='upper right', fontsize=10)
ax3.grid(True, alpha=0.3, axis='y')

# 添加數值標籤
for bar in bars1:
    height = bar.get_height()
    ax3.annotate(f'{height:.1f}%',
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3),
                textcoords="offset points",
                ha='center', va='bottom', fontsize=9)

for bar in bars2:
    height = bar.get_height()
    ax3.annotate(f'{height:.1f}%',
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 3),
                textcoords="offset points",
                ha='center', va='bottom', fontsize=9)

# 子圖4: 通膨分析摘要
ax4 = axes[1, 1]
ax4.axis('off')

inflation_summary = """
╔══════════════════════════════════════════════════════════════════════╗
║                    12月通膨數據分析摘要                               ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  🇺🇸 美國:                                                           ║
║     • 12月CPI: 核心商品與服務均偏軟                                   ║
║     • 短線低於市場與JPM原估                                          ║
║     • 關稅驅動的核心商品黏性暫緩                                      ║
║     • 期貨市場預期: 全年降息兩碼，6月首次                             ║
║                                                                      ║
║  🇪🇺 歐元區:                                                         ║
║     • 12月HICP: 總體1.9%，核心2.3%                                   ║
║     • 商品極弱(0.4%)，服務仍偏高(3.4%)                               ║
║     • 薪資談判顯示未來幾季薪資放緩                                    ║
║     • 能源是上行尾部風險主角                                         ║
║     • 油價年初上漲+5%，HICP預測上調0.1ppt                            ║
║                                                                      ║
║  ⚠️ 風險因素:                                                        ║
║     • 美國Fed獨立性疑慮                                              ║
║     • 若降息>2碼: 可能是衰退(利率下)或政治干預(通膨上)               ║
║     • 伊朗局勢若升級，能源衝擊風險                                    ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
"""

ax4.text(0.5, 0.5, inflation_summary, transform=ax4.transAxes, fontsize=10,
         verticalalignment='center', horizontalalignment='center',
         fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='lightcyan', alpha=0.8))

plt.tight_layout()
plt.savefig('inflation_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n圖表2: 通膨數據分析已生成")

---
## 3. Fed 政策利率預期與市場定價

In [ ]:
# Fed Funds Futures 相關 Ticker
fed_futures_tickers = {
    'FF_Feb26': 'FFG6 Comdty',
    'FF_Mar26': 'FFH6 Comdty',
    'FF_Jun26': 'FFM6 Comdty',
    'FF_Sep26': 'FFU6 Comdty',
    'FF_Dec26': 'FFZ6 Comdty',
}

# ECB 利率相關
ecb_tickers = {
    'ECB_Deposit_Rate': 'EURR002W Index',
    'ESTR': 'ESTRON Index'
}

print("央行政策利率 Ticker 定義完成")

In [ ]:
# 圖表3: 央行政策利率預期
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Fed 利率路徑預期
ax1 = axes[0]

meetings = ['2026/01', '2026/03', '2026/05', '2026/06', '2026/07', '2026/09', '2026/11', '2026/12']
current_rate = 4.375  # 當前 Fed Funds 區間中點

# 市場隱含利率路徑
market_implied = [4.375, 4.375, 4.375, 4.125, 4.125, 4.125, 3.875, 3.875]
# JPM 預測路徑
jpm_forecast = [4.375, 4.375, 4.375, 4.125, 4.125, 3.875, 3.875, 3.625]
# 鷹派情境
hawkish_scenario = [4.375, 4.375, 4.375, 4.375, 4.375, 4.375, 4.375, 4.375]

x = np.arange(len(meetings))

ax1.plot(x, market_implied, 'b-o', linewidth=2.5, markersize=8, label='市場隱含')
ax1.plot(x, jpm_forecast, 'g--s', linewidth=2, markersize=7, label='JPM 預測', alpha=0.8)
ax1.plot(x, hawkish_scenario, 'r-.^', linewidth=1.5, markersize=6, label='鷹派情境', alpha=0.6)

ax1.fill_between(x, [r - 0.125 for r in market_implied], 
                 [r + 0.125 for r in market_implied], alpha=0.2, color='blue')

ax1.set_xlabel('FOMC 會議', fontsize=12)
ax1.set_ylabel('Fed Funds Rate (%)', fontsize=12)
ax1.set_title('2026年 Fed 利率路徑預期', fontsize=14, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(meetings, rotation=45)
ax1.legend(loc='upper right', fontsize=10)
ax1.grid(True, alpha=0.3)
ax1.set_ylim(3.5, 4.6)

# 標註降息時點
ax1.annotate('首次降息\n(6月)', xy=(3, 4.125), xytext=(3, 4.35),
            arrowprops=dict(arrowstyle='->', color='blue'),
            fontsize=10, ha='center', color='blue')

# ECB 利率路徑
ax2 = axes[1]

ecb_meetings = ['2026/01', '2026/03', '2026/04', '2026/06', '2026/07', '2026/09', '2026/10', '2026/12']
ecb_current = 3.0  # 當前 ECB 存款利率

ecb_market_implied = [3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0, 3.0]
ecb_consensus = [3.0, 2.75, 2.75, 2.5, 2.5, 2.5, 2.25, 2.25]

x2 = np.arange(len(ecb_meetings))

ax2.plot(x2, ecb_market_implied, 'b-o', linewidth=2.5, markersize=8, label='市場隱含 (不降息)')
ax2.plot(x2, ecb_consensus, 'g--s', linewidth=2, markersize=7, label='共識預測', alpha=0.8)

ax2.set_xlabel('ECB 會議', fontsize=12)
ax2.set_ylabel('ECB Deposit Rate (%)', fontsize=12)
ax2.set_title('2026年 ECB 利率路徑預期', fontsize=14, fontweight='bold')
ax2.set_xticks(x2)
ax2.set_xticklabels(ecb_meetings, rotation=45)
ax2.legend(loc='upper right', fontsize=10)
ax2.grid(True, alpha=0.3)
ax2.set_ylim(2.0, 3.5)

# 標註
ax2.annotate('市場預期\n不降息', xy=(4, 3.0), xytext=(4, 3.25),
            arrowprops=dict(arrowstyle='->', color='blue'),
            fontsize=10, ha='center', color='blue')

plt.tight_layout()
plt.savefig('central_bank_policy.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n圖表3: 央行政策利率預期已生成")

---
## 4. 消費支出 (PCE) 與經濟數據分析

In [ ]:
# 消費與經濟數據 Ticker
economic_tickers = {
    # PCE 相關
    'PCE_Total': 'PCE CUR$ Index',
    'PCE_Goods': 'PCENGSA Index',
    'PCE_Services': 'PCESVA Index',
    
    # 收入與儲蓄
    'Personal_Income': 'PITL Index',
    'Disposable_Income': 'PIDY Index',
    'Savings_Rate': 'PIDSSAVR Index',
    
    # 勞動市場
    'Nonfarm_Payrolls': 'NFP TCH Index',
    'Unemployment_Rate': 'USURTOT Index',
    'Avg_Hourly_Earnings': 'AHE YOY% Index',
    
    # 進出口
    'Trade_Balance': 'USTBTOT Index',
    'Imports': 'USTBIMP Index',
    'Exports': 'USTBEXP Index'
}

print("經濟數據 Ticker 定義完成")

In [ ]:
# 圖表4: PCE 消費支出與經濟數據
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 季度數據
quarters = ['2024Q1', '2024Q2', '2024Q3', '2024Q4', '2025Q1', '2025Q2', '2025Q3', '2025Q4']

# PCE 商品消費成長 (季增年率)
pce_goods_growth = [1.8, 2.5, 2.2, 1.5, 2.8, 2.4, 2.9, 3.1]
pce_services_growth = [3.2, 3.5, 3.0, 2.8, 2.9, 3.1, 2.8, 2.6]
pce_total_growth = [2.6, 3.1, 2.7, 2.3, 2.9, 2.8, 2.8, 2.8]

# 子圖1: PCE 成長趨勢
ax1 = axes[0, 0]
x = np.arange(len(quarters))
width = 0.25

bars1 = ax1.bar(x - width, pce_goods_growth, width, label='商品', color='steelblue', alpha=0.8)
bars2 = ax1.bar(x, pce_services_growth, width, label='服務', color='coral', alpha=0.8)
bars3 = ax1.bar(x + width, pce_total_growth, width, label='總體', color='forestgreen', alpha=0.8)

ax1.axhline(y=3.1, color='red', linestyle='--', linewidth=2, alpha=0.7)
ax1.annotate('4Q25商品: 3.1%\n(優於預期)', xy=(7, 3.1), xytext=(5.5, 3.5),
            arrowprops=dict(arrowstyle='->', color='red'),
            fontsize=10, color='red', fontweight='bold')

ax1.set_xlabel('季度', fontsize=12)
ax1.set_ylabel('季增年率 (%)', fontsize=12)
ax1.set_title('PCE 消費支出成長率', fontsize=14, fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(quarters, rotation=45)
ax1.legend(loc='upper left', fontsize=10)
ax1.grid(True, alpha=0.3, axis='y')

# 子圖2: 儲蓄率與可支配所得
ax2 = axes[0, 1]

months = pd.date_range(start='2024-01-01', periods=24, freq='MS')
savings_rate = [5.8, 5.6, 5.4, 5.2, 5.0, 4.8, 4.7, 4.5, 4.4, 4.3, 4.2, 4.1,
                4.0, 3.9, 3.8, 3.7, 3.6, 3.5, 3.5, 3.4, 3.4, 3.3, 3.3, 3.2]

disposable_income_growth = [3.2, 3.1, 3.0, 2.9, 2.8, 2.7, 2.6, 2.5, 2.4, 2.3, 2.2, 2.1,
                            2.0, 2.0, 1.9, 1.9, 1.8, 1.8, 1.8, 1.7, 1.7, 1.7, 1.6, 1.6]

ax2_twin = ax2.twinx()

line1, = ax2.plot(months, savings_rate, 'b-o', linewidth=2, markersize=4, label='儲蓄率')
line2, = ax2_twin.plot(months, disposable_income_growth, 'r-s', linewidth=2, markersize=4, label='可支配所得成長')

ax2.set_xlabel('日期', fontsize=12)
ax2.set_ylabel('儲蓄率 (%)', fontsize=12, color='blue')
ax2_twin.set_ylabel('可支配所得成長 (%)', fontsize=12, color='red')
ax2.set_title('儲蓄率 vs 可支配所得成長', fontsize=14, fontweight='bold')

ax2.tick_params(axis='y', labelcolor='blue')
ax2_twin.tick_params(axis='y', labelcolor='red')

lines = [line1, line2]
labels = [l.get_label() for l in lines]
ax2.legend(lines, labels, loc='upper right', fontsize=10)
ax2.grid(True, alpha=0.3)
ax2.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax2.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.setp(ax2.xaxis.get_majorticklabels(), rotation=45)

# 標註
ax2.annotate('儲蓄率持續下降\n→消費支撐力道', xy=(months[20], 3.4), xytext=(months[14], 4.5),
            arrowprops=dict(arrowstyle='->', color='blue'),
            fontsize=10, color='blue')

# 子圖3: 進口與庫存分析
ax3 = axes[1, 0]

import_data = [100, 102, 105, 108, 112, 115, 118, 120, 122, 120, 118, 115,
               112, 110, 108, 106, 105, 104, 103, 102, 101, 100, 99, 98]
inventory_data = [95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106,
                  107, 108, 109, 110, 110, 110, 110, 110, 110, 110, 110, 110]

ax3_twin = ax3.twinx()

line1, = ax3.plot(months, import_data, 'b-', linewidth=2, label='進口指數')
ax3.fill_between(months, import_data, alpha=0.3, color='blue')
line2, = ax3_twin.plot(months, inventory_data, 'r-', linewidth=2, label='庫存指數')

ax3.set_xlabel('日期', fontsize=12)
ax3.set_ylabel('進口指數', fontsize=12, color='blue')
ax3_twin.set_ylabel('庫存指數', fontsize=12, color='red')
ax3.set_title('進口下降但庫存維持 → 廠商對終端消費不悲觀', fontsize=14, fontweight='bold')

ax3.tick_params(axis='y', labelcolor='blue')
ax3_twin.tick_params(axis='y', labelcolor='red')

lines = [line1, line2]
labels = [l.get_label() for l in lines]
ax3.legend(lines, labels, loc='upper right', fontsize=10)
ax3.grid(True, alpha=0.3)
ax3.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax3.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.setp(ax3.xaxis.get_majorticklabels(), rotation=45)

# 添加註解框
textstr = '進口↓ + 庫存穩定\n\n可能原因:\n1. 提前拉貨，庫存足夠\n2. 國內廠商補上缺口'
props = dict(boxstyle='round', facecolor='wheat', alpha=0.8)
ax3.text(0.02, 0.98, textstr, transform=ax3.transAxes, fontsize=10,
        verticalalignment='top', bbox=props)

# 子圖4: 經濟數據摘要
ax4 = axes[1, 1]
ax4.axis('off')

economic_summary = """
╔══════════════════════════════════════════════════════════════════════╗
║                    4Q25 經濟數據分析摘要                              ║
╠══════════════════════════════════════════════════════════════════════╣
║                                                                      ║
║  📊 消費支出 (PCE):                                                  ║
║     • 4Q25商品消費成長3.1%，優於預期                                  ║
║     • 消費支出強勁，即使可支配所得沒有增加                             ║
║     • 儲蓄率持續下降 (目前約3.2%)                                     ║
║                                                                      ║
║  💼 勞動市場:                                                        ║
║     • 工資成長疲軟                                                   ║
║     • 私部門就業仍有增加                                              ║
║     • 整體就業市場保持韌性                                            ║
║                                                                      ║
║  📦 進出口與庫存:                                                    ║
║     • 整體進口下降但庫存不減                                          ║
║     • 兩種可能解釋:                                                  ║
║       1. 提前拉貨，庫存足夠終端消費                                   ║
║       2. 國內廠商正在補上進口商缺口                                   ║
║     • 兩種情況都顯示廠商對終端消費並不悲觀                             ║
║                                                                      ║
║  🎯 前瞻展望:                                                        ║
║     • OBBBA法案稅收優惠有望刺激2026上半年支出                         ║
║     • 儲蓄率下降可能限制消費持續性                                    ║
║     • 需關注勞動市場是否進一步惡化                                    ║
║                                                                      ║
╚══════════════════════════════════════════════════════════════════════╝
"""

ax4.text(0.5, 0.5, economic_summary, transform=ax4.transAxes, fontsize=10,
         verticalalignment='center', horizontalalignment='center',
         fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='honeydew', alpha=0.8))

plt.tight_layout()
plt.savefig('pce_economic_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n圖表4: PCE消費支出與經濟數據分析已生成")

---
## 5. 綜合數據儀表板

In [ ]:
# 綜合儀表板 - BQL 實時數據查詢
def create_dashboard_query():
    """
    建立綜合儀表板的 BQL 查詢
    """
    # 所有需要監控的 Ticker
    all_tickers = [
        # 殖利率
        'USGG2YR Index', 'USGG5YR Index', 'USGG10YR Index', 'USGG30YR Index',
        # 通膨預期
        'USGGBE05 Index', 'USGGBE10 Index',
        # 政策利率
        'FDTR Index',  # Fed Funds Target Rate
        # 能源
        'CL1 Comdty', 'CO1 Comdty'
    ]
    
    # BQL 查詢
    fields = {
        'last_price': bq.data.px_last(),
        'change_1d': bq.data.chg_pct_1d(),
        'change_1w': bq.data.day_to_day_chg_pct(days=5),
        'change_1m': bq.data.day_to_day_chg_pct(days=21)
    }
    
    return bql.Request(all_tickers, fields)

print("儀表板查詢函數已建立")

In [ ]:
# 圖表5: 綜合監控儀表板
fig = plt.figure(figsize=(20, 14))

# 使用 GridSpec 進行更靈活的佈局
from matplotlib.gridspec import GridSpec
gs = GridSpec(3, 3, figure=fig, hspace=0.3, wspace=0.3)

# ===== 左上: 殖利率曲線快照 =====
ax1 = fig.add_subplot(gs[0, 0])
tenors_short = ['2Y', '5Y', '10Y', '30Y']
current_yields = [4.05, 4.10, 4.35, 4.65]
yesterday_yields = [4.08, 4.12, 4.38, 4.68]

x = np.arange(len(tenors_short))
width = 0.35
bars1 = ax1.bar(x - width/2, current_yields, width, label='今日', color='steelblue')
bars2 = ax1.bar(x + width/2, yesterday_yields, width, label='昨日', color='lightsteelblue')

ax1.set_ylabel('殖利率 (%)')
ax1.set_title('美債殖利率快照', fontweight='bold')
ax1.set_xticks(x)
ax1.set_xticklabels(tenors_short)
ax1.legend()
ax1.grid(True, alpha=0.3, axis='y')

# ===== 中上: 曲線利差 =====
ax2 = fig.add_subplot(gs[0, 1])
spreads = ['2s10s', '5s30s', '10s30s']
spread_values = [30, 55, 30]
spread_changes = [2, -3, -2]

colors = ['green' if v > 0 else 'red' for v in spread_changes]
bars = ax2.bar(spreads, spread_values, color='steelblue', alpha=0.7)

for bar, change in zip(bars, spread_changes):
    height = bar.get_height()
    color = 'green' if change > 0 else 'red'
    sign = '+' if change > 0 else ''
    ax2.annotate(f'{sign}{change}bp',
                xy=(bar.get_x() + bar.get_width() / 2, height),
                xytext=(0, 5),
                textcoords="offset points",
                ha='center', fontsize=11, fontweight='bold', color=color)

ax2.set_ylabel('利差 (bp)')
ax2.set_title('關鍵曲線利差', fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')

# ===== 右上: 通膨預期 =====
ax3 = fig.add_subplot(gs[0, 2])

breakeven_tenors = ['5Y BE', '10Y BE']
us_breakeven = [2.35, 2.42]
eu_breakeven = [2.15, 2.28]

x = np.arange(len(breakeven_tenors))
width = 0.35
ax3.bar(x - width/2, us_breakeven, width, label='美國', color='steelblue')
ax3.bar(x + width/2, eu_breakeven, width, label='歐元區', color='darkorange')

ax3.axhline(y=2.0, color='green', linestyle='--', linewidth=2, alpha=0.7)
ax3.set_ylabel('Breakeven (%)')
ax3.set_title('通膨預期比較', fontweight='bold')
ax3.set_xticks(x)
ax3.set_xticklabels(breakeven_tenors)
ax3.legend()
ax3.grid(True, alpha=0.3, axis='y')

# ===== 中左: 政策利率預期 =====
ax4 = fig.add_subplot(gs[1, 0])

# 降息機率條形圖
meetings_cut = ['3月', '5月', '6月', '9月', '12月']
cut_probabilities = [5, 15, 65, 75, 90]

colors = ['lightcoral' if p < 50 else 'lightgreen' for p in cut_probabilities]
bars = ax4.barh(meetings_cut, cut_probabilities, color=colors, alpha=0.8)
ax4.axvline(x=50, color='black', linestyle='--', linewidth=1)

for bar, prob in zip(bars, cut_probabilities):
    ax4.annotate(f'{prob}%',
                xy=(prob + 2, bar.get_y() + bar.get_height()/2),
                va='center', fontsize=10, fontweight='bold')

ax4.set_xlabel('降息機率 (%)')
ax4.set_title('Fed 降息機率 (累積)', fontweight='bold')
ax4.set_xlim(0, 100)
ax4.grid(True, alpha=0.3, axis='x')

# ===== 中中: 能源價格 =====
ax5 = fig.add_subplot(gs[1, 1])

energy_dates = pd.date_range(end=datetime.now(), periods=60, freq='B')
wti_prices = 72 + np.cumsum(np.random.randn(60) * 0.5)
brent_prices = 76 + np.cumsum(np.random.randn(60) * 0.5)

ax5.plot(energy_dates, wti_prices, 'b-', linewidth=2, label='WTI')
ax5.plot(energy_dates, brent_prices, 'r-', linewidth=2, label='Brent')

ax5.set_ylabel('價格 ($/桶)')
ax5.set_title('原油價格走勢 (3個月)', fontweight='bold')
ax5.legend(loc='upper left')
ax5.grid(True, alpha=0.3)
ax5.xaxis.set_major_formatter(mdates.DateFormatter('%m/%d'))
ax5.xaxis.set_major_locator(mdates.WeekdayLocator(interval=2))

# 標註年初漲幅
ax5.annotate('年初+5%', xy=(energy_dates[-1], wti_prices[-1]), 
            xytext=(energy_dates[-15], wti_prices[-1]+3),
            arrowprops=dict(arrowstyle='->', color='blue'),
            fontsize=10, color='blue')

# ===== 中右: 經濟指標熱力圖 =====
ax6 = fig.add_subplot(gs[1, 2])

indicators = ['GDP成長', 'PCE消費', '失業率', '通膨', '製造業PMI']
current_values = [2.5, 3.1, 4.1, 2.0, 49.2]
expectations = [2.3, 2.8, 4.0, 2.2, 50.0]
surprises = [v - e for v, e in zip(current_values, expectations)]

# 創建數據矩陣
data_matrix = np.array([surprises]).T

im = ax6.imshow(data_matrix, cmap='RdYlGn', aspect='auto', vmin=-0.5, vmax=0.5)
ax6.set_yticks(np.arange(len(indicators)))
ax6.set_yticklabels(indicators)
ax6.set_xticks([0])
ax6.set_xticklabels(['驚喜度'])

# 添加數值標籤
for i, (v, s) in enumerate(zip(current_values, surprises)):
    sign = '+' if s > 0 else ''
    ax6.text(0, i, f'{v}\n({sign}{s:.1f})', ha='center', va='center', fontsize=10)

ax6.set_title('經濟指標 vs 預期', fontweight='bold')
plt.colorbar(im, ax=ax6, label='驚喜度', shrink=0.8)

# ===== 下方: 策略總結 =====
ax7 = fig.add_subplot(gs[2, :])
ax7.axis('off')

summary_text = """
╔════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════╗
║                                              2026年1月26日 宏觀策略總結                                                     ║
╠════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════╣
║                                                                                                                            ║
║  🎯 核心觀點:                                                                                                              ║
║     1. 美債曲線: 看弱30年期，看好腹部(5Y-10Y)，做多5s/30s與10s/30s利差                                                       ║
║     2. 利率預期: Fed全年降息兩碼(6月首次)，ECB維持不變                                                                        ║
║     3. 通膨風險: 美國通膨風險低，歐洲需關注能源上行尾部風險                                                                    ║
║     4. 消費前景: 4Q25 PCE商品消費3.1%優於預期，但儲蓄率下降需關注持續性                                                        ║
║                                                                                                                            ║
║  ⚠️ 關鍵風險:                                                                                                              ║
║     • Fed獨立性: 若降息>2碼，可能反映衰退(殖利率下)或政治干預(通膨上)                                                          ║
║     • 能源價格: 油價年初+5%，伊朗局勢若升級將帶來更大衝擊                                                                      ║
║     • 財政赤字: 持續壓力影響長端殖利率                                                                                        ║
║                                                                                                                            ║
║  📈 推薦交易:                                                                                                              ║
║     • 做多 5s/30s 利差 (進場: 55bp, 目標: 75bp, 止損: 45bp)                                                                  ║
║     • 做多 10s/30s 利差 (進場: 30bp, 目標: 45bp, 止損: 22bp)                                                                 ║
║     • 考慮接收短端利率 (2Y區間)，等待通膨數據確認                                                                              ║
║                                                                                                                            ║
╚════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════════╝
"""

ax7.text(0.5, 0.5, summary_text, transform=ax7.transAxes, fontsize=11,
         verticalalignment='center', horizontalalignment='center',
         fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.9))

plt.suptitle('Bloomberg BQNT 宏觀策略儀表板', fontsize=16, fontweight='bold', y=0.98)
plt.savefig('macro_dashboard.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n圖表5: 綜合監控儀表板已生成")

---
## 6. BQL 實時數據查詢範例

In [ ]:
# BQL 實時查詢範例 - 殖利率曲線
def fetch_live_yield_curve():
    """
    從 Bloomberg 獲取實時殖利率曲線數據
    """
    tickers = [
        'USGG3M Index',   # 3個月
        'USGG6M Index',   # 6個月
        'USGG12M Index',  # 1年
        'USGG2YR Index',  # 2年
        'USGG3YR Index',  # 3年
        'USGG5YR Index',  # 5年
        'USGG7YR Index',  # 7年
        'USGG10YR Index', # 10年
        'USGG20YR Index', # 20年
        'USGG30YR Index'  # 30年
    ]
    
    # 基本查詢
    query = bql.Request(
        tickers,
        {
            'yield': bq.data.px_last(),
            'change': bq.data.chg_pct_1d(),
            'high_52w': bq.data.px_high(start='-1Y'),
            'low_52w': bq.data.px_low(start='-1Y')
        }
    )
    
    try:
        response = bq.execute(query)
        df = pd.DataFrame()
        for field in ['yield', 'change', 'high_52w', 'low_52w']:
            df[field] = response[field].df()['value']
        return df
    except Exception as e:
        print(f"查詢錯誤: {e}")
        return None

# 執行查詢
print("BQL 殖利率曲線查詢範例:")
print("-" * 50)
print("""bql.Request(
    ['USGG2YR Index', 'USGG5YR Index', 'USGG10YR Index', 'USGG30YR Index'],
    {
        'yield': bq.data.px_last(),
        'change': bq.data.chg_pct_1d()
    }
)""")

In [ ]:
# BQL 實時查詢範例 - 曲線利差計算
def calculate_spreads():
    """
    計算關鍵曲線利差
    """
    # 使用 BQL 計算利差
    # 方法1: 直接查詢利差 Ticker
    spread_tickers = [
        'USYC2Y10 Index',  # 2s10s
        'USYC5Y30 Index',  # 5s30s (如果存在)
    ]
    
    # 方法2: 自行計算
    yield_tickers = {
        '2Y': 'USGG2YR Index',
        '5Y': 'USGG5YR Index',
        '10Y': 'USGG10YR Index',
        '30Y': 'USGG30YR Index'
    }
    
    query = bql.Request(
        list(yield_tickers.values()),
        {'yield': bq.data.px_last()}
    )
    
    try:
        response = bq.execute(query)
        yields = {}
        for tenor, ticker in yield_tickers.items():
            yields[tenor] = response['yield'].df().loc[ticker, 'value']
        
        # 計算利差 (以 bp 為單位)
        spreads = {
            '2s10s': (yields['10Y'] - yields['2Y']) * 100,
            '5s30s': (yields['30Y'] - yields['5Y']) * 100,
            '10s30s': (yields['30Y'] - yields['10Y']) * 100
        }
        
        return pd.Series(spreads, name='Spread (bp)')
    except Exception as e:
        print(f"查詢錯誤: {e}")
        return None

print("BQL 曲線利差計算範例:")
print("-" * 50)
print("""# 查詢各年期殖利率後自行計算
spreads = {
    '2s10s': (yields['10Y'] - yields['2Y']) * 100,
    '5s30s': (yields['30Y'] - yields['5Y']) * 100,
    '10s30s': (yields['30Y'] - yields['10Y']) * 100
}""")

In [ ]:
# BQL 實時查詢範例 - 經濟數據
def fetch_economic_data():
    """
    從 Bloomberg 獲取關鍵經濟數據
    """
    economic_tickers = {
        # 通膨
        'US_CPI': 'CPI YOY Index',
        'US_Core_CPI': 'CPUPAXFE Index',
        'EU_HICP': 'ECCPEST Index',
        
        # 消費
        'US_PCE': 'PCE DEFY Index',
        'US_Retail_Sales': 'RSTAMOM Index',
        
        # 勞動市場
        'US_Unemployment': 'USURTOT Index',
        'US_NFP': 'NFP TCH Index',
        
        # 儲蓄率
        'US_Savings_Rate': 'PIDSSAVR Index'
    }
    
    query = bql.Request(
        list(economic_tickers.values()),
        {
            'value': bq.data.px_last(),
            'prev_value': bq.data.px_last(dates='-1M')
        }
    )
    
    return query

print("BQL 經濟數據查詢範例:")
print("-" * 50)
print("""bql.Request(
    ['CPI YOY Index', 'PCE DEFY Index', 'USURTOT Index', 'PIDSSAVR Index'],
    {
        'value': bq.data.px_last(),
        'prev_value': bq.data.px_last(dates='-1M')
    }
)""")

---
## 7. 輸出與報告生成

In [ ]:
# 生成報告摘要
print("="*80)
print("                    20260126 宏觀策略分析報告                    ")
print("="*80)
print()
print("【殖利率曲線觀點】")
print("  • 看弱 30 年期: 赤字壓力 + Fed 獨立性風險")
print("  • 看好腹部區域 (5Y-10Y): 相對價值較佳")
print("  • 推薦交易: 做多 5s/30s 與 10s/30s 利差")
print()
print("【通膨展望】")
print("  • 美國: 12月CPI核心商品與服務均偏軟，通膨風險低")
print("  • 歐洲: 薪資放緩有利通膨下行，但需關注能源風險")
print("  • 市場預期: Fed 全年降息兩碼 (6月首次)，ECB 維持不變")
print()
print("【消費與經濟】")
print("  • 4Q25 PCE 商品消費成長 3.1%，優於預期")
print("  • 儲蓄率持續下降，消費持續性需關注")
print("  • 進口下降但庫存穩定 → 廠商對終端消費不悲觀")
print()
print("【關鍵風險】")
print("  • Fed 獨立性疑慮: 若降息>2碼可能反映衰退或政治干預")
print("  • 能源上行風險: 油價年初+5%，伊朗局勢需密切關注")
print()
print("="*80)
print("生成圖表: ")
print("  1. yield_curve_analysis.png - 殖利率曲線分析")
print("  2. inflation_analysis.png - 通膨數據比較")
print("  3. central_bank_policy.png - 央行政策利率預期")
print("  4. pce_economic_analysis.png - PCE消費與經濟數據")
print("  5. macro_dashboard.png - 綜合監控儀表板")
print("="*80)

In [ ]:
# 數據表格輸出
print("\n【關鍵數據摘要表】\n")

# 殖利率數據
yield_summary = pd.DataFrame({
    '年期': ['2Y', '5Y', '10Y', '30Y'],
    '當前殖利率': ['4.05%', '4.10%', '4.35%', '4.65%'],
    '日變化': ['-3bp', '-2bp', '-3bp', '-3bp'],
    '週變化': ['-8bp', '-5bp', '-2bp', '+2bp'],
    '觀點': ['中性', '看多', '看多', '看空']
})
print("美債殖利率:")
print(yield_summary.to_string(index=False))

print("\n" + "-"*60 + "\n")

# 曲線利差數據
spread_summary = pd.DataFrame({
    '利差': ['2s10s', '5s30s', '10s30s'],
    '當前': ['30bp', '55bp', '30bp'],
    '目標': ['40bp', '75bp', '45bp'],
    '建議': ['觀望', '做多', '做多']
})
print("曲線利差策略:")
print(spread_summary.to_string(index=False))

print("\n" + "-"*60 + "\n")

# 通膨比較
inflation_summary = pd.DataFrame({
    '項目': ['總體通膨', '核心通膨', '商品', '服務'],
    '美國': ['2.0%', '2.3%', '0.2%', '3.2%'],
    '歐元區': ['1.9%', '2.3%', '0.4%', '3.4%']
})
print("12月通膨數據比較:")
print(inflation_summary.to_string(index=False))